# MCP (Model Context Protocol) — server + client walkthrough

This notebook builds two things on top of `mcp_server.py` (same folder):

1. **Raw protocol client** — connect over stdio, do the `initialize` handshake, list tools, call a tool directly. No LLM involved, just the wire protocol.
2. **Claude-powered client** — same server, but now Claude decides which tool to call and when, via the `anthropic[mcp]` conversion helpers + `tool_runner`.

`mcp_server.py` is a small FastMCP app exposing three tools: `calculate`, `remember`, `recall`. It has no idea Claude exists — it just answers `tools/list` / `tools/call` over stdio.

## 0. Concept

**MCP (Model Context Protocol)** is an open protocol, created by Anthropic (Nov 2024), that standardizes how an LLM application connects to external tools, data, and prompts. Before MCP, every app that wanted Claude to read files, query a database, or hit an API wrote its own bespoke integration. MCP defines one protocol so a server written once (e.g. a "GitHub MCP server") works with any MCP-compatible client (Claude Desktop, Claude Code, your own app, other agent frameworks) — the same idea as USB-C standardizing device connectors, or LSP standardizing how editors talk to language servers.

### Architecture

```
┌─────────────┐         MCP (JSON-RPC 2.0)         ┌─────────────┐
│  MCP Host   │◄──────────────────────────────────►│  MCP Server │
│ (your app)  │         over stdio or HTTP          │ (tool/data  │
│             │                                     │  provider)  │
│  ┌────────┐ │                                     └─────────────┘
│  │  MCP   │ │
│  │ Client │ │  1 client per server connection
│  └────────┘ │
└─────────────┘
```

- **Host** — the application the user interacts with (Claude Desktop, your CLI app, an agent framework). Owns the LLM, the UI, and orchestration.
- **Client** — lives inside the host, maintains a 1:1 connection to one server, speaks the protocol.
- **Server** — a separate process (or remote endpoint) that exposes capabilities: **tools** (functions the model can call), **resources** (data the model can read), **prompts** (reusable prompt templates). A server knows nothing about the LLM — it just answers protocol requests.

A host can talk to multiple servers at once (multiple client instances) — e.g. Claude Desktop connecting to a filesystem server, a GitHub server, and a Slack server simultaneously.

### Wire protocol

MCP messages are **JSON-RPC 2.0** — requests have `id`/`method`/`params`, responses have a matching `id` with `result` or `error`, and one-way messages are `notifications` (no `id`, no response expected).

**Transports** (how the bytes move):
- **stdio** — the server is a local subprocess; the client writes JSON-RPC to its stdin and reads from its stdout. Simplest, used for local tools — this is what our `mcp_server.py` uses below.
- **Streamable HTTP** — the server is a remote endpoint; the client POSTs JSON-RPC and can receive an SSE stream back for server-initiated messages. Used for hosted/remote servers.

**Lifecycle:**
1. **`initialize`** — the client sends its protocol version + capabilities; the server responds with its own version + capabilities (which of tools/resources/prompts it supports) + server info.
2. **`notifications/initialized`** — the client confirms; the connection is now live.
3. Normal operation — the client calls `tools/list`, `tools/call`, `resources/list`, `resources/read`, `prompts/list`, `prompts/get`, etc.
4. Either side can send `notifications/*` for things like "tool list changed."

**The primitive that matters most — tools:**
- `tools/list` → the server returns each tool's `name`, `description`, `inputSchema` (JSON Schema) — this is the same shape Claude's Messages API tool definitions use, which is why MCP tools convert to Claude tools almost 1:1.
- `tools/call` with `{name, arguments}` → the server executes and returns a `content` array (text/image/etc.) or an error.

The cells below build exactly this: a server exposing tools over stdio, a raw client that speaks the protocol directly, and a Claude-powered client that lets the model decide which tool to call.

In [1]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Tells the client how to launch the server: run `python mcp_server.py`
# as a subprocess and speak MCP over its stdin/stdout.
server_params = StdioServerParameters(
    command="python",
    args=["mcp_server.py"],
)

In [2]:
from dotenv import load_dotenv

# Loads ANTHROPIC_API_KEY (and anything else) from the .env file in this
# folder into the environment — only needed for the Claude-powered section
# below; the raw protocol cells don't touch the API at all.
load_dotenv()

True

## 1. Raw protocol client

`stdio_client(server_params)` spawns the server subprocess and gives us its read/write streams. `ClientSession` wraps those streams with the JSON-RPC framing. `session.initialize()` performs the MCP handshake (protocol version + capability exchange) before anything else is allowed. Only after that can we call `list_tools()` / `call_tool()`.

In [4]:
# --- Connect, handshake, discover tools ---
async def list_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()  # the MCP handshake
            result = await session.list_tools()
            for t in result.tools:
                print(f"- {t.name}: {t.description}")
            return result

_ = await list_tools()

- calculate: Evaluate a basic arithmetic expression (+, -, *, /, parentheses) and return the result.

    Args:
        expression: e.g. "12 * (3 + 4)"
    
- remember: Store a short note in the server's in-memory list for later recall.

    Args:
        note: The text to remember.
    
- recall: Return every note stored so far via the remember tool.


In [5]:
# --- Call tools directly, no LLM involved ---
async def raw_tool_calls():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            r1 = await session.call_tool("calculate", {"expression": "12 * (3 + 4)"})
            print("calculate ->", r1.content[0].text)

            r2 = await session.call_tool(
                "remember",
                {"note": "MCP session state only lives as long as the connection is open"},
            )
            print("remember  ->", r2.content[0].text)

            r3 = await session.call_tool("recall", {})
            print("recall    ->", r3.content[0].text)

await raw_tool_calls()

calculate -> 84
remember  -> Stored note #1: 'MCP session state only lives as long as the connection is open'
recall    -> 1. MCP session state only lives as long as the connection is open


## 2. Claude-powered client

Now Claude decides which tool to call and when, instead of us hardcoding it. `anthropic.lib.tools.mcp.async_mcp_tool` converts each MCP tool (name, description, JSON Schema) into the shape `client.beta.messages.tool_runner()` expects, and wires its execution straight to `session.call_tool(...)` — so when Claude emits a `tool_use`, the runner forwards it through this same MCP session to the server.

Requires `ANTHROPIC_API_KEY` to be set (or an active `ant auth login` profile).

In [6]:
from anthropic import AsyncAnthropic
from anthropic.lib.tools.mcp import async_mcp_tool

client = AsyncAnthropic()


async def chat(user_message: str) -> str:
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()

            # Same MCP tools from before, now wrapped for Claude's tool_runner.
            claude_tools = [async_mcp_tool(t, session) for t in tools_result.tools]

            # tool_runner() is sync — it just builds the runner; iterate with `async for`.
            runner = client.beta.messages.tool_runner(
                model="claude-sonnet-5",
                max_tokens=1024,
                tools=claude_tools,
                messages=[{"role": "user", "content": user_message}],
            )

            final_text = ""
            async for message in runner:
                for block in message.content:
                    if block.type == "text":
                        final_text = block.text
            return final_text

In [8]:
answer = await chat(
    "Remember that my favorite language is Python, then calculate 15 * 7, "
    "and tell me both the stored note and the calculation result."
)
print(answer)

### Session lifecycle gotcha

Each call to `chat()` above opens a brand-new `stdio_client` — a fresh server subprocess and a fresh session. So the note stored in one `chat()` call **will not** be visible to `recall()` in a separate `chat()` call: state lives on the connection/session, not across independent client invocations. (A real app would keep one session open across a whole conversation instead of reconnecting per turn.) Try it below — `recall` will come back empty even though we just stored a note above.

In [ ]:
answer2 = await chat("What notes do you have stored? Use the recall tool to check.")
print(answer2)